### Spark Architecture

In [0]:
###
Spark is a distributed computing platform
Every Spark application is a distributed application itself
Spark application needs a cluster
Cluster technologies:
    Hadoop YARN
    Kubernetes
    Apache Mesos
    Spark Standalone
###

Cluster is a pool of physical computers

When we want to run a spark application:
    We will submit it using spark-submit command 
    it will send a request to YARN RM (Resources manager)
    The YARN RM will craete a application master (AM container) on a worker node and start my application main() in the container
    A container is an isolated virtual runtime environemnt, it comes with a CPU and memory allocation
    suppose we have 16 CPU cores and 64GB RAM in one executor. YARN may assin 4 CPU cores and 16GB RAM out of it to the AM container.

Lets check inside the container and see what happens there:
    The container runs the main() of the application 
    The main method could be a pyspark application or a scala application.
    [Spark is written in SCALA (SCALA is a JVM language) and it always runs in JVM]
    To run Python code in spark the developer craeted a JAVA wrapper on top of the SCALA code. And then they created a Pytgon Wrapper on top of JAVA Wrapper. And this Python wrapper is known as PySpark

For pyspark application:
    Suppose we have python code in my main method (PySpark). 
    This python code is design to start a JAVA main method internally.
    So my pyspark application will start a JVM application. Once we have the JVM application the python wrapper will call the JVM wrapper using the py4j connection.
    py4j allows a python application to call a JAVA application. And thats how pyspark works.
    It always starts the JVM application and calls python APIs in the JVM.
    The actuall spark application is alwys a SCALA application running in JVM.
    But PySprk calls Java Wrapper using py4j. And the Java Wrapper runs the SCALA code inside JVM. 
    Pyspark main () - PySpark Driver
    JVM main () - Application Driver

    SPARK CORE - JAVA WRAPPER - PYTHON WRAPPER

    If you wrote a PySpark application you will have a PySpark Driver and a Application Driver. But if you wrote a SCALA application you will have a Application Driver. One will always have a Application Driver



In [0]:
Apllication written in SCALA/JAVA will have Application Driver and JVM inside each executor.
Application written in Pyspakr will have Pyspark Driver and Application Driver and JVM inside each executor.
If you are using additional Python libraries apart from Pyspark (even if UDFs) we will have a python worker inside all worker nodes. JVM + Python Worker.
If you are using some python libraries which doesnot have a Java Wrapper, you will need a Python worker (python runtime) in each Executor.

### Submit Modes

In [0]:
>>Spark Submit allows you to submit your spark application to the cluster in 2 modes: Cluster and Client 
>>Code:
  spark-submit --master yarn --deploy-mode cluster
  spark-submit --master yarn --deploy-mode client

>>Spark has two deploy modes: client and cluster. In client mode, the driver runs on the local machine, while in cluster mode, the driver runs inside the cluster. Azure Databricks uses cluster mode internally, as the driver always runs on the cluster’s driver node, even when executing notebooks interactively.


### JOBS, STAGES, TASKS etc

In [0]:
>>Most of the Spark APIs can be classified into 2 catagories. 
1.Transformations:
    
2.Action:
    
>>Example of some spark APIs which are neither transformation nor actions:
    Configuration & Session APIs:
        SparkSession.builder(), spark.conf.set(), spark.sparkContext, setLogLevel().....Because they don’t operate on a dataset—they configure the engine.
    Metadata / Schema Inspection APIs:
        df.printSchema(), df.schema, df.columns, df.dtypes....They don’t compute the dataset—just read metadata already known.
    Logical Plan / Debugging APIs:
        df.explain(), df.queryExecution.. They analyze the plan, not execute it.
    Caching / Persistence APIs:
        df.cache(), df.persist(), df.unpersist()
        .....cache() is lazy. It only marks the DataFrame to be cached. Actual caching happens only when an action runs. It doesn’t change the data logically.
    Dependency / Lineage Control APIs:
        df.checkpoint()
    Partitioning / Execution Control APIs:
        repartition(), coalesce()...They don’t change data content
They change data distribution. So, Still transformations (they create new RDD/DataFrame). But conceptually different from map, filter
        
    

In [0]:
1. Transformation:
    a. Narrow dependency:
        can run in parallel on each data partitions independent with other partitions. select(), filter(), withColumn(), drop()
    b. Wide Dependency:
        Parformed after grouping data from multiple partitions. groupBy(), join(), cube(), agg(), rollup(), repartition(). These requires grouping on some keys and then apply some transformation.
2. Actions:
    Actions trigger some work(Job).
    read(), write(), collect(), take() and count()
    All spark actions triggers one or more jobs.
        
    

In [0]:
>>Lets devide the code into blocks. First block starts from beginning and ends while I call an action.
>>So each block will now considered as one spark JOB. Each action creates a spark job.
>>The excutors must run these jobs, this shouldnot be done by Driver.
>>Driver should break the jobs into smaller tasks.

>>Spark driver will create a logical query plan for each spark job.
Logical plan example:
    

![image_1774205481817.png](./image_1774205481817.png "image_1774205481817.png")

In [0]:
>>The driver will next plan to breaek this Logical plan into stages.
>>Driver will look for WIDE DEPENDENCY Transformations for breaking into stages. (Here repartition and group by)

![image_1774205615356.png](./image_1774205615356.png "image_1774205615356.png")

In [0]:
>>Each stage can have one or more narrow tranformations and the last operation of the stages will be wide transformation.
>>Spark cannont run those stages in parallel. One should be finished before proceeding next. Because O/P for 1st stg is I/P for next stg.
>>Write Exchange and read exchange and Exchange buffer and it required shuffle/short which are heavy and expensive.

![image_1774206001998.png](./image_1774206001998.png "image_1774206001998.png")

![image_1774206031060.png](./image_1774206031060.png "image_1774206031060.png")

In [0]:
>>Each slot performs one TASK. Slots are nothing but CPU cores

![image_1774206336797.png](./image_1774206336797.png "image_1774206336797.png")

### SPARK SQL Engine and Query Planning

In [0]:
>>The above explaination about JOBS and stages works with Dataframe API code (or Dataset API code). 
>>Spark SQL API, each SQL expression are known as single JOB.
>>Spark code is nothing but a sequence of Spark Jobs. And each spark job represents a logical query plan.

![image_1774207216817.png](./image_1774207216817.png "image_1774207216817.png")

![image_1774207384778.png](./image_1774207384778.png "image_1774207384778.png")

In [0]:
>>The above are done by SPARK SQL Engine, whether you run in SPARK or SQL
>>Thats why Spark is also called as compiler. It analyse the requiremnt and craetes and plan from logical to physical using spark SQL engine and utilizes the best plan.

#Performance and Applied Understanding:

In [0]:
PySpark acts as a Python interface that communicates with Spark’s JVM-based engine via Py4J, where the actual execution of standard operations like filter, join, groupBy, and Spark SQL happens inside the JVM using heap memory. The Python code itself does not run in the JVM; instead, it runs in separate Python worker processes. Whenever Python-specific logic is involved—such as UDFs, Pandas UDFs, or external libraries—the computation is executed in these Python processes, and their memory usage is accounted as memory overhead rather than heap. In Spark, heap memory is used for core data processing within the JVM, while memory overhead covers everything outside it, including Python workers, JVM internals, shuffle buffers, and off-heap storage. Together they form the total executor memory, and insufficient overhead can lead to failures even if heap memory is available.

In [0]:
Overhead memory in Spark is used for all non-heap operations such as Python worker processes, data transfer buffers, shuffle operations, JVM internals, and off-heap execution, making it essential for stable execution especially in PySpark and shuffle-heavy workloads.

![image_1774210318900.png](./image_1774210318900.png "image_1774210318900.png")

![image_1774210421499.png](./image_1774210421499.png "image_1774210421499.png")

default overhead is 384 MB. If 10% is lower then 384 it will take 384.If higher, then it will take higher number 

In [0]:
Total Memory =
   Heap Memory (JVM)
 + Overhead Memory
      ├── Python Memory
      ├── JVM Internals
      ├── Shuffle Buffers
      └── Native / Off-heap (partly)

### Executor Memory Breakdown

![image_1774211268453.png](./image_1774211268453.png "image_1774211268453.png")

Here Spark memory is actual memory pool

In [0]:
>>The 60%–40% split represents only JVM heap memory, where caching and execution happen inside Spark memory, and user memory is for JVM-level objects; however, Python UDFs in PySpark do not use this heap space and instead consume memory from the overhead, which is not shown in this diagram.
>>The 40% user memory in heap stores JVM-level user objects, metadata, and Scala/Java UDF logic, while Python UDFs in PySpark do not use this space and instead consume memory from overhead outside the JVM.

![image_1774212747393.png](./image_1774212747393.png "image_1774212747393.png")

![image_1774212769676.png](./image_1774212769676.png "image_1774212769676.png")

![image_1774212785667.png](./image_1774212785667.png "image_1774212785667.png")

![image_1774212800798.png](./image_1774212800798.png "image_1774212800798.png")

![image_1774212831133.png](./image_1774212831133.png "image_1774212831133.png")

### AQE - Adaptive Query Execution

In [0]:
>>spark.sql.shuffle.partitions = 10 / setting number of output partitions. For changing size dataset, it is almost impossible to predict the number here.
>>If we configure a small number, we may get OOM exception and if it is too big, we will have more unneccesay burden in Spark Task Scheduler. 
>>Sperk Shuffle sort has a critical impact on Spark query performance. Blank partition are empty but spark will still trigger them as Task. The empty taks will do nothing and finish in mili seconds. But spark scheduler needs some time for scheduling and monitoring those tasks. Although the overhead is small we will be having unnecesasy overhead here.
>>To handle this we can set a large number of shuffle partitions in the beginneing and enable AQE. AQE will dynamically handle the shuffle partitions determine the best number during runtime and set it for the next and combine or coalesce the small partitions.

In [0]:
>>Spark Execution plan is built before the execution triggered so the size of the Data is still unknown
>>AQE COMPUTES THE STATISTICS ON THE SHUFFLE DATA DURING RUNTIME.

![image_1774214681414.png](./image_1774214681414.png "image_1774214681414.png")

![image_1774217054080.png](./image_1774217054080.png "image_1774217054080.png")

In [0]:
>>Partition pruning → Spark skips reading unnecessary partitions based on filter conditions on partition columns

>>Predicate pushdown → Filters are pushed to the data source (like Parquet/DB) so only required data is read

>>Dynamic partition pruning → Spark filters partitions of a large (fact) table at runtime using values from a smaller (dimension) table during a join, avoiding unnecessary data reads

>>Achieving it → partition the fact table + use join on partition column + enable AQE and broadcast the dimension table (spark.sql.optimizer.dynamicPartitionPruning.enabled=true)

In [0]:
>>cache() and persist() will cache the DF in the Executor Storage Memory pool 
>>Both the above are lazy transformations. They do not cache until an action is triggered.

![image_1774220010410.png](./image_1774220010410.png "image_1774220010410.png")

In [0]:
>>For offHeap, we should first add some offHeap memory in the Executor memory.
>>Deserialized → whether cached data is stored as objects (fast, more memory) or serialized (compact, slower)
>>replication → number of copies of cached data across executors for fault tolerance
>>df.unpersist() will uncache .. we donot have any uncache().
>>When we are accessing large DF accross the spark actions, use caching.

### Repartition and Coalesce

In [0]:
>>Repartition is a wide dependency tranformation. So it causes shuffle/sort of the DF
>>If we do not provide the repartion size it will be controlled by spark.sql.shufflePartion size provided earlier. It can be override by supplying the number in repartition. 
>>repartition(numPartitions, *cols) → hash-based partitioning; data is randomly distributed (or by hash of columns) across partitions
>>repartitionByRange(numPartitions, *cols) → range-based partitioning; data is distributed based on sorted value ranges (uses sampling)
>>use repartition → when you need even data distribution, fix skew, or increase parallelism (general use)
>>use repartitionByRange → when doing range queries, sorting, or operations like joins on ordered data (better for range-based operations)

![image_1774220992697.png](./image_1774220992697.png "image_1774220992697.png")

In [0]:
>>Repartition on a column name does not gurentee the uniform partition value.
>>Repartition causes shuffle/short.

In [0]:
>>We should use coalesce() when we want to reduce the partitions. 
>>It conbines local partitions only and can cause skewed partitions if we reduce the number drastically.
>>No shuffle-sort so not expensive.
>>If we try to increase the number of partition using coalesce() it will do nothing.

### Dataframe Hints

![image_1774221753845.png](./image_1774221753845.png "image_1774221753845.png")

![image_1774221999020.png](./image_1774221999020.png "image_1774221999020.png")

![image_1774222014028.png](./image_1774222014028.png "image_1774222014028.png")

### Broadcase Variables

In [0]:
>>Broadcast variable → a small read-only dataset sent once to all executors to avoid repeated data transfer and improve performance (commonly used in broadcast joins and lookup scenarios)

broadcast_var = sc.broadcast(lookup_dict)

>>Used in small lookup data (few MBs), avoiding shuffle in joins, repeated access inside transformations.
>>Not to use in large datasets (can cause memory issues)

### Accumulator

In [0]:
>>An accumulator is a variable used to aggregate values across executors (write-only in workers, read in driver).
>>Where it is used:
    counting events (e.g., bad records, null values)
    tracking metrics during processing
    debugging / monitoring job behavior

>>How it works:
    Executors → add/update value
    Driver → reads final result
++++++
acc = spark.sparkContext.longAccumulator("counter")
df.foreach(lambda row: acc.add(1))
print(acc.value)
++++++

![image_1774222829578.png](./image_1774222829578.png "image_1774222829578.png")

### Speculative Execution 

In [0]:
>>By default it is false. We can enable it using spark.speculation = true.
>>It is usefull if we are expecting an VM can be faulty (Hardware) where some task can run slow. 
>>If a task is runnig very long, SP-Task(duplicate) will run on parallel and if any of these finishes first, other will be killed.

![image_1774223240722.png](./image_1774223240722.png "image_1774223240722.png")